# AI Challenge 2025 Video Extraction Pipeline

This notebook builds a Milestone 2 sample data pipeline for 10 videos from Kaggle dataset `aresusayhi/ai-challenge-2025`.

Outputs follow the Research handoff contract:
- keyframes uploaded to MinIO, with local backup under `baseline_output/keyframes/`
- `embeddings.npy` and `vectors.npy`
- `frame_mapping.csv`
- `frame_ids.txt`
- `annotations.jsonl`, `annotations.json`, and compatibility alias `annations.json`
- `dataset_manifest.yaml`
- `model_info.json`

AIThena alignment: shot-like keyframe extraction, redundancy removal, visual vectors, OCR/caption/object/ASR annotation fields, and storage-ready metadata for vector search plus metadata retrieval.

## 1. Optional dependency install

Set `INSTALL_DEPS = True` if your notebook kernel does not already have the packages.

In [ ]:
INSTALL_DEPS = False

if INSTALL_DEPS:
    import sys
    import subprocess

    packages = [
        'kaggle',
        'minio',
        'opencv-python',
        'pillow',
        'numpy',
        'pandas',
        'pyyaml',
        'tqdm',
        'sentence-transformers',
    ]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *packages])

## 2. Configuration

Kaggle credentials can be provided by `KAGGLE_USERNAME` and `KAGGLE_KEY`, or by `~/.kaggle/kaggle.json`.

MinIO defaults are compatible with a common local dev setup. Override them with environment variables when your Docker stack uses different values.

In [ ]:
import hashlib
import importlib.util
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from PIL import Image

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(items, **kwargs):
        return items


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'docs').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start


REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / 'notebooks' / 'Test'
WORK_DIR = NOTEBOOK_DIR / 'video_extraction_work'
RAW_DATA_DIR = WORK_DIR / 'raw' / 'ai-challenge-2025'
OUTPUT_DIR = NOTEBOOK_DIR / 'baseline_output'
LOCAL_KEYFRAME_DIR = OUTPUT_DIR / 'keyframes'

CONFIG = {
    'kaggle_slug': 'aresusayhi/ai-challenge-2025',
    'run_kaggle_download': True,
    'force_kaggle_download': False,
    'max_videos': 10,
    'sample_strategy': 'first',  # first or random
    'random_seed': 2026,
    'max_keyframes_per_video': 12,
    'shot_scan_interval_sec': 1.0,
    'min_shot_length_sec': 2.0,
    'shot_diff_threshold': 0.22,
    'dedupe_similarity_threshold': 0.985,
    'jpeg_quality': 92,
    'embedding_backend': 'sentence-transformers-clip',
    'embedding_model_name': 'clip-ViT-B-32',
    'embedding_batch_size': 32,
    'fallback_embedding_dim': 512,
    'allow_histogram_fallback': True,
    'enable_ocr': False,
    'enable_captioning': False,
    'upload_to_minio': True,
    'strict_minio': False,
    'minio_endpoint': os.getenv('MINIO_ENDPOINT', 'localhost:9000'),
    'minio_access_key': os.getenv('MINIO_ACCESS_KEY', 'minioadmin'),
    'minio_secret_key': os.getenv('MINIO_SECRET_KEY', 'minioadmin'),
    'minio_bucket': os.getenv('MINIO_BUCKET', 'aic-keyframes'),
    'minio_prefix': os.getenv('MINIO_PREFIX', 'milestone2/keyframes'),
    'minio_secure': os.getenv('MINIO_SECURE', 'false').lower() in {'1', 'true', 'yes'},
}

for directory in [WORK_DIR, RAW_DATA_DIR, OUTPUT_DIR, LOCAL_KEYFRAME_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('Repo root:', REPO_ROOT)
print('Output dir:', OUTPUT_DIR)

## 3. Dependency check

In [ ]:
PACKAGE_CHECKS = {
    'cv2': 'opencv-python',
    'kaggle': 'kaggle',
    'minio': 'minio',
    'sentence_transformers': 'sentence-transformers',
    'PIL': 'pillow',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'yaml': 'pyyaml',
}

missing = []
for module_name, package_name in PACKAGE_CHECKS.items():
    if importlib.util.find_spec(module_name) is None:
        missing.append(package_name)

if missing:
    print('Missing packages:', sorted(set(missing)))
    print('Set INSTALL_DEPS=True in the first code cell, run it, then restart the kernel if needed.')
else:
    print('Core packages are available.')

## 4. Kaggle download and video scan

In [ ]:
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.m4v'}


def has_kaggle_credentials():
    kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
    return bool(os.getenv('KAGGLE_USERNAME') and os.getenv('KAGGLE_KEY')) or kaggle_json.exists()


def download_kaggle_dataset(config):
    target_dir = RAW_DATA_DIR
    existing_files = [p for p in target_dir.rglob('*') if p.is_file()]
    if existing_files and not config['force_kaggle_download']:
        print(f'Skip Kaggle download because {target_dir} already has {len(existing_files)} files.')
        return target_dir

    if importlib.util.find_spec('kaggle') is None:
        raise ImportError('Package kaggle is missing. Set INSTALL_DEPS=True or run: pip install kaggle')
    if not has_kaggle_credentials():
        raise RuntimeError(
            'Kaggle credentials not found. Set KAGGLE_USERNAME/KAGGLE_KEY or place kaggle.json in ~/.kaggle/.'
        )

    from kaggle.api.kaggle_api_extended import KaggleApi

    target_dir.mkdir(parents=True, exist_ok=True)
    api = KaggleApi()
    api.authenticate()
    print(f'Downloading Kaggle dataset {config["kaggle_slug"]} to {target_dir} ...')
    api.dataset_download_files(config['kaggle_slug'], path=str(target_dir), unzip=True, quiet=False)
    return target_dir


def scan_videos(root_dir):
    root_dir = Path(root_dir)
    videos = sorted(
        p for p in root_dir.rglob('*')
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    )
    return videos


def safe_video_code(path, seen=None):
    seen = seen if seen is not None else set()
    stem = re.sub(r'[^A-Za-z0-9_\-]+', '_', Path(path).stem).strip('_')
    if not stem:
        stem = hashlib.sha1(str(path).encode('utf-8')).hexdigest()[:10]
    code = stem
    if code in seen:
        suffix = hashlib.sha1(str(path).encode('utf-8')).hexdigest()[:8]
        code = f'{stem}_{suffix}'
    seen.add(code)
    return code


def sample_videos(videos, config):
    videos = list(videos)
    if config['sample_strategy'] == 'random':
        rng = random.Random(config['random_seed'])
        rng.shuffle(videos)
    return videos[: int(config['max_videos'])]


def build_dataset_manifest(all_videos, selected_videos, config):
    seen = set()
    selected = []
    for video_path in selected_videos:
        selected.append({
            'video_code': safe_video_code(video_path, seen),
            'path': str(video_path),
            'relative_path': str(video_path.relative_to(RAW_DATA_DIR)) if str(video_path).startswith(str(RAW_DATA_DIR)) else str(video_path),
            'size_bytes': video_path.stat().st_size,
        })

    return {
        'dataset_slug': config['kaggle_slug'],
        'created_at': datetime.now(timezone.utc).isoformat(),
        'raw_data_dir': str(RAW_DATA_DIR),
        'num_videos_found': len(all_videos),
        'num_videos_selected': len(selected_videos),
        'sample_strategy': config['sample_strategy'],
        'selected_videos': selected,
    }


if CONFIG['run_kaggle_download']:
    dataset_dir = download_kaggle_dataset(CONFIG)
else:
    dataset_dir = RAW_DATA_DIR

all_video_paths = scan_videos(dataset_dir)
selected_video_paths = sample_videos(all_video_paths, CONFIG)
dataset_manifest = build_dataset_manifest(all_video_paths, selected_video_paths, CONFIG)

manifest_path = OUTPUT_DIR / 'dataset_manifest.yaml'
with manifest_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(dataset_manifest, f, allow_unicode=True, sort_keys=False)

print(f'Found {len(all_video_paths)} videos. Selected {len(selected_video_paths)} videos.')
print('Manifest:', manifest_path)
dataset_manifest['selected_videos'][:3]

## 5. MinIO helpers

In [ ]:
def get_minio_client(config):
    if not config['upload_to_minio']:
        return None
    if importlib.util.find_spec('minio') is None:
        message = 'Package minio is missing. Set INSTALL_DEPS=True or run: pip install minio'
        if config['strict_minio']:
            raise ImportError(message)
        print('MinIO disabled:', message)
        return None

    from minio import Minio

    endpoint_raw = config['minio_endpoint']
    secure = bool(config['minio_secure']) or endpoint_raw.startswith('https://')
    endpoint = endpoint_raw.replace('http://', '').replace('https://', '').strip('/')
    try:
        client = Minio(
            endpoint,
            access_key=config['minio_access_key'],
            secret_key=config['minio_secret_key'],
            secure=secure,
        )
        bucket = config['minio_bucket']
        if not client.bucket_exists(bucket):
            client.make_bucket(bucket)
        print(f'MinIO ready: bucket={bucket}, endpoint={endpoint}')
        return client
    except Exception as exc:
        if config['strict_minio']:
            raise
        print(f'MinIO unavailable, keeping local keyframes only: {exc}')
        return None


def upload_keyframe_to_minio(client, local_path, video_code, config):
    if client is None:
        return {
            'minio_bucket': None,
            'minio_key': None,
            'minio_uri': None,
            'presigned_url': None,
        }

    bucket = config['minio_bucket']
    prefix = config['minio_prefix'].strip('/')
    object_name = f'{prefix}/{video_code}/{Path(local_path).name}'
    client.fput_object(bucket, object_name, str(local_path), content_type='image/jpeg')

    presigned_url = None
    try:
        presigned_url = client.presigned_get_object(bucket, object_name, expires=timedelta(days=7))
    except Exception:
        pass

    return {
        'minio_bucket': bucket,
        'minio_key': object_name,
        'minio_uri': f's3://{bucket}/{object_name}',
        'presigned_url': presigned_url,
    }

## 6. Keyframe extraction

AIThena uses AutoShot and keeps first/middle/last frames per shot, then removes near-duplicates with semantic similarity. This notebook uses a lightweight shot-change detector so the pipeline is runnable in a notebook, while keeping the same handoff shape.

In [ ]:
def require_cv2():
    if importlib.util.find_spec('cv2') is None:
        raise ImportError('OpenCV is missing. Set INSTALL_DEPS=True or run: pip install opencv-python')
    import cv2
    return cv2


def cosine_similarity(a, b, eps=1e-12):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps))


def frame_signature(frame_bgr):
    cv2 = require_cv2()
    small = cv2.resize(frame_bgr, (32, 32), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    vector = gray.reshape(-1)
    vector = vector - vector.mean()
    norm = np.linalg.norm(vector)
    return vector / norm if norm > 0 else vector


def read_video_metadata(video_path):
    cv2 = require_cv2()
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open video: {video_path}')

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    if fps <= 0 or math.isnan(fps):
        fps = 25.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    cap.release()

    return {
        'fps': fps,
        'frame_count': frame_count,
        'duration_ms': int((frame_count / fps) * 1000) if frame_count > 0 else None,
        'width': width,
        'height': height,
    }


def read_frame(video_path, frame_idx):
    cv2 = require_cv2()
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open video: {video_path}')
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame = cap.read()
    cap.release()
    if not ok or frame is None:
        return None
    return frame


def detect_shot_segments(video_path, metadata, config):
    cv2 = require_cv2()
    frame_count = int(metadata['frame_count'])
    fps = float(metadata['fps'])
    if frame_count <= 1:
        return [(0, 0)]

    scan_step = max(1, int(fps * float(config['shot_scan_interval_sec'])))
    min_gap = max(1, int(fps * float(config['min_shot_length_sec'])))
    threshold = float(config['shot_diff_threshold'])

    cap = cv2.VideoCapture(str(video_path))
    boundaries = [0]
    previous_signature = None

    for frame_idx in range(0, frame_count, scan_step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
        if not ok or frame is None:
            continue
        signature = frame_signature(frame)
        if previous_signature is not None:
            diff = float(np.mean(np.abs(signature - previous_signature)))
            if diff >= threshold and (frame_idx - boundaries[-1]) >= min_gap:
                boundaries.append(frame_idx)
        previous_signature = signature

    cap.release()

    if boundaries[-1] != frame_count - 1:
        boundaries.append(frame_count - 1)

    segments = []
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        if end > start:
            segments.append((int(start), int(end)))
    return segments or [(0, frame_count - 1)]


def representative_indices(segments, max_keyframes):
    candidates = []
    for start, end in segments:
        middle = int((start + end) / 2)
        candidates.extend([start, middle, end])
    candidates = sorted(set(candidates))
    if len(candidates) <= max_keyframes:
        return candidates
    positions = np.linspace(0, len(candidates) - 1, max_keyframes).round().astype(int)
    return [candidates[i] for i in sorted(set(positions.tolist()))]


def segment_id_for_frame(frame_idx, segments):
    for idx, (start, end) in enumerate(segments):
        if start <= frame_idx <= end:
            return f'shot_{idx:04d}'
    return 'shot_unknown'


def save_frame_jpeg(frame_bgr, output_path, jpeg_quality=92):
    cv2 = require_cv2()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    params = [int(cv2.IMWRITE_JPEG_QUALITY), int(jpeg_quality)]
    ok = cv2.imwrite(str(output_path), frame_bgr, params)
    if not ok:
        raise RuntimeError(f'Failed to save image: {output_path}')


def extract_keyframes_for_video(video_path, video_code, config):
    metadata = read_video_metadata(video_path)
    segments = detect_shot_segments(video_path, metadata, config)
    candidates = representative_indices(segments, int(config['max_keyframes_per_video']))
    fps = float(metadata['fps'])

    video_keyframe_dir = LOCAL_KEYFRAME_DIR / video_code
    records = []
    kept_signatures = []

    for frame_idx in candidates:
        frame = read_frame(video_path, frame_idx)
        if frame is None:
            continue
        signature = frame_signature(frame)
        if any(cosine_similarity(signature, previous) >= float(config['dedupe_similarity_threshold']) for previous in kept_signatures):
            continue

        frame_id = f'{video_code}_{frame_idx:06d}'
        image_path = video_keyframe_dir / f'{frame_id}.jpg'
        save_frame_jpeg(frame, image_path, config['jpeg_quality'])
        kept_signatures.append(signature)

        timestamp_ms = int((frame_idx / fps) * 1000) if fps > 0 else None
        records.append({
            'frame_id': frame_id,
            'video_code': video_code,
            'video_path': str(video_path),
            'frame_idx': int(frame_idx),
            'timestamp_ms': timestamp_ms,
            'timestamp_sec': round(timestamp_ms / 1000.0, 3) if timestamp_ms is not None else None,
            'image_path': str(image_path),
            'image_path_relative': str(image_path.relative_to(OUTPUT_DIR)),
            'shot_id': segment_id_for_frame(frame_idx, segments),
            'video_fps': fps,
            'video_frame_count': int(metadata['frame_count']),
            'video_duration_ms': metadata['duration_ms'],
            'width': int(metadata['width']),
            'height': int(metadata['height']),
        })

    return records, metadata, segments

## 7. Embedding and annotation helpers

In [ ]:
def fallback_image_embedding(image_path, dim=512):
    image = Image.open(image_path).convert('RGB')
    arr = np.asarray(image.resize((32, 32), Image.Resampling.BICUBIC), dtype=np.float32) / 255.0

    hist_parts = []
    for channel in range(3):
        hist, _ = np.histogram(arr[:, :, channel], bins=64, range=(0.0, 1.0), density=True)
        hist_parts.append(hist.astype(np.float32))

    gray = np.asarray(image.convert('L').resize((20, 16), Image.Resampling.BICUBIC), dtype=np.float32).reshape(-1) / 255.0
    vector = np.concatenate([*hist_parts, gray.astype(np.float32)], axis=0)
    if vector.shape[0] < dim:
        vector = np.pad(vector, (0, dim - vector.shape[0]))
    vector = vector[:dim].astype(np.float32)
    norm = np.linalg.norm(vector)
    return vector / norm if norm > 0 else vector


class ImageEmbedder:
    def __init__(self, config):
        self.config = config
        self.backend = 'histogram-fallback'
        self.model_name = 'color-histogram-grayscale-512'
        self.model = None
        self.device = 'cuda' if importlib.util.find_spec('torch') is not None and self._cuda_available() else 'cpu'

        if config['embedding_backend'] == 'sentence-transformers-clip':
            self._try_load_sentence_transformer_clip()

    def _cuda_available(self):
        try:
            import torch
            return torch.cuda.is_available()
        except Exception:
            return False

    def _try_load_sentence_transformer_clip(self):
        if importlib.util.find_spec('sentence_transformers') is None:
            if self.config['allow_histogram_fallback']:
                print('sentence-transformers is missing. Using histogram fallback embeddings.')
                return
            raise ImportError('sentence-transformers is missing.')

        try:
            from sentence_transformers import SentenceTransformer

            self.model_name = self.config['embedding_model_name']
            self.model = SentenceTransformer(self.model_name, device=self.device)
            self.backend = 'sentence-transformers-clip'
            print(f'Loaded embedding model {self.model_name} on {self.device}.')
        except Exception as exc:
            if self.config['allow_histogram_fallback']:
                print(f'Could not load {self.config["embedding_model_name"]}: {exc}')
                print('Using histogram fallback embeddings so the pipeline can still produce vectors.')
                self.backend = 'histogram-fallback'
                self.model_name = 'color-histogram-grayscale-512'
                self.model = None
                return
            raise

    def encode(self, image_paths):
        image_paths = [Path(p) for p in image_paths]
        if self.backend == 'sentence-transformers-clip' and self.model is not None:
            images = [Image.open(path).convert('RGB') for path in image_paths]
            vectors = self.model.encode(
                images,
                batch_size=int(self.config['embedding_batch_size']),
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=True,
            )
            return vectors.astype(np.float32)

        vectors = [fallback_image_embedding(path, int(self.config['fallback_embedding_dim'])) for path in tqdm(image_paths, desc='Fallback embeddings')]
        return np.stack(vectors).astype(np.float32)


def load_ocr_engine(config):
    if not config['enable_ocr']:
        return None
    try:
        from paddleocr import PaddleOCR
        return PaddleOCR(use_angle_cls=True, lang='vi')
    except Exception as exc:
        print(f'OCR disabled because PaddleOCR could not be loaded: {exc}')
        return None


def run_ocr(ocr_engine, image_path):
    if ocr_engine is None:
        return ''
    try:
        result = ocr_engine.ocr(str(image_path), cls=True)
        texts = []
        for page in result or []:
            for item in page or []:
                if len(item) >= 2 and len(item[1]) >= 1:
                    texts.append(str(item[1][0]))
        return ' '.join(texts).strip()
    except Exception as exc:
        print(f'OCR failed for {image_path}: {exc}')
        return ''


def load_captioner(config):
    if not config['enable_captioning']:
        return None
    try:
        import torch
        from transformers import BlipForConditionalGeneration, BlipProcessor

        model_id = 'Salesforce/blip-image-captioning-base'
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        processor = BlipProcessor.from_pretrained(model_id)
        model = BlipForConditionalGeneration.from_pretrained(model_id).to(device)
        return {'processor': processor, 'model': model, 'device': device, 'model_id': model_id}
    except Exception as exc:
        print(f'Captioning disabled because BLIP could not be loaded: {exc}')
        return None


def generate_caption(captioner, image_path, record):
    if captioner is None:
        return f'Keyframe from {record["video_code"]} at {record["timestamp_sec"]} seconds.'

    try:
        import torch

        image = Image.open(image_path).convert('RGB')
        inputs = captioner['processor'](image, return_tensors='pt').to(captioner['device'])
        with torch.no_grad():
            output = captioner['model'].generate(**inputs, max_new_tokens=30)
        return captioner['processor'].decode(output[0], skip_special_tokens=True)
    except Exception as exc:
        print(f'Caption failed for {image_path}: {exc}')
        return f'Keyframe from {record["video_code"]} at {record["timestamp_sec"]} seconds.'


def make_annotation(record, embedding_index, ocr_text, caption, model_versions):
    return {
        'frame_id': record['frame_id'],
        'video_code': record['video_code'],
        'frame_idx': record['frame_idx'],
        'timestamp_ms': record['timestamp_ms'],
        'timestamp_sec': record['timestamp_sec'],
        'shot_id': record['shot_id'],
        'image_path': record['image_path_relative'],
        'minio_bucket': record.get('minio_bucket'),
        'minio_key': record.get('minio_key'),
        'minio_uri': record.get('minio_uri'),
        'presigned_url': record.get('presigned_url'),
        'embedding_index': int(embedding_index),
        'ocr_text': ocr_text,
        'asr_text': '',
        'caption': caption,
        'objects': [],
        'scene': '',
        'metadata': {
            'source_video_path': record['video_path'],
            'video_fps': record['video_fps'],
            'video_frame_count': record['video_frame_count'],
            'video_duration_ms': record['video_duration_ms'],
            'width': record['width'],
            'height': record['height'],
        },
        'model_versions': model_versions,
    }

## 8. End-to-end pipeline

In [ ]:
def clean_previous_outputs():
    paths_to_clean = [
        OUTPUT_DIR / 'embeddings.npy',
        OUTPUT_DIR / 'vectors.npy',
        OUTPUT_DIR / 'frame_mapping.csv',
        OUTPUT_DIR / 'frame_ids.txt',
        OUTPUT_DIR / 'annotations.jsonl',
        OUTPUT_DIR / 'annotations.json',
        OUTPUT_DIR / 'annations.json',
        OUTPUT_DIR / 'model_info.json',
        OUTPUT_DIR / 'upload_manifest.json',
    ]
    for path in paths_to_clean:
        if path.exists():
            path.unlink()


def extract_and_upload_keyframes(selected_items, config):
    minio_client = get_minio_client(config)
    all_records = []
    video_reports = []

    for item in tqdm(selected_items, desc='Videos'):
        video_path = Path(item['path'])
        video_code = item['video_code']
        print(f'Extracting {video_code}: {video_path}')
        records, metadata, segments = extract_keyframes_for_video(video_path, video_code, config)

        for record in records:
            upload_info = upload_keyframe_to_minio(minio_client, record['image_path'], video_code, config)
            record.update(upload_info)

        all_records.extend(records)
        video_reports.append({
            'video_code': video_code,
            'video_path': str(video_path),
            'metadata': metadata,
            'num_detected_segments': len(segments),
            'num_keyframes': len(records),
        })

    upload_manifest = {
        'created_at': datetime.now(timezone.utc).isoformat(),
        'minio_enabled': minio_client is not None,
        'minio_bucket': config['minio_bucket'] if minio_client is not None else None,
        'minio_prefix': config['minio_prefix'] if minio_client is not None else None,
        'videos': video_reports,
    }
    with (OUTPUT_DIR / 'upload_manifest.json').open('w', encoding='utf-8') as f:
        json.dump(upload_manifest, f, ensure_ascii=False, indent=2)

    return all_records, upload_manifest


def export_outputs(records, embeddings, embedder, config):
    if len(records) != embeddings.shape[0]:
        raise ValueError(f'Record/vector mismatch: {len(records)} records vs {embeddings.shape[0]} vectors')

    embeddings = embeddings.astype(np.float32)
    np.save(OUTPUT_DIR / 'embeddings.npy', embeddings)
    np.save(OUTPUT_DIR / 'vectors.npy', embeddings)

    frame_ids = [record['frame_id'] for record in records]
    with (OUTPUT_DIR / 'frame_ids.txt').open('w', encoding='utf-8') as f:
        f.write('\n'.join(frame_ids) + '\n')

    mapping_rows = []
    for index, record in enumerate(records):
        mapping_rows.append({
            'frame_id': record['frame_id'],
            'video_code': record['video_code'],
            'frame_idx': record['frame_idx'],
            'timestamp_ms': record['timestamp_ms'],
            'image_path': record['image_path_relative'],
            'shot_id': record['shot_id'],
            'embedding_index': index,
            'minio_bucket': record.get('minio_bucket'),
            'minio_key': record.get('minio_key'),
            'minio_uri': record.get('minio_uri'),
        })
    mapping_df = pd.DataFrame(mapping_rows)
    mapping_df.to_csv(OUTPUT_DIR / 'frame_mapping.csv', index=False, encoding='utf-8')

    ocr_engine = load_ocr_engine(config)
    captioner = load_captioner(config)
    model_versions = {
        'keyframe_extraction': 'simple-shot-first-middle-last-dedupe',
        'embedding': embedder.model_name,
        'embedding_backend': embedder.backend,
        'ocr': 'paddleocr-vi' if ocr_engine is not None else 'disabled',
        'caption': captioner['model_id'] if captioner is not None else 'template-caption',
        'asr': 'not-run',
        'object_detection': 'not-run',
        'scene': 'not-run',
    }

    annotations = []
    for index, record in enumerate(tqdm(records, desc='Annotations')):
        image_path = Path(record['image_path'])
        ocr_text = run_ocr(ocr_engine, image_path)
        caption = generate_caption(captioner, image_path, record)
        annotations.append(make_annotation(record, index, ocr_text, caption, model_versions))

    jsonl_path = OUTPUT_DIR / 'annotations.jsonl'
    with jsonl_path.open('w', encoding='utf-8') as f:
        for annotation in annotations:
            f.write(json.dumps(annotation, ensure_ascii=False) + '\n')

    annotations_doc = {
        'schema_version': 'milestone2.sample.v1',
        'created_at': datetime.now(timezone.utc).isoformat(),
        'dataset_slug': config['kaggle_slug'],
        'embedding_file': 'embeddings.npy',
        'frame_mapping_file': 'frame_mapping.csv',
        'items': annotations,
    }
    for filename in ['annotations.json', 'annations.json']:
        with (OUTPUT_DIR / filename).open('w', encoding='utf-8') as f:
            json.dump(annotations_doc, f, ensure_ascii=False, indent=2)

    model_info = {
        'model_name': embedder.model_name,
        'provider': embedder.backend,
        'embedding_dim': int(embeddings.shape[1]),
        'normalized': True,
        'checkpoint': embedder.model_name,
        'num_vectors': int(embeddings.shape[0]),
        'dtype': str(embeddings.dtype),
        'paper_alignment': {
            'source': 'AIThena.pdf',
            'target_system': 'AIthena-Vision data preprocessing baseline',
            'notes': [
                'Extract keyframes from videos.',
                'Remove visually redundant frames.',
                'Generate visual feature vectors for vector search.',
                'Keep OCR, ASR, caption, object, and scene fields for metadata retrieval.',
            ],
        },
    }
    with (OUTPUT_DIR / 'model_info.json').open('w', encoding='utf-8') as f:
        json.dump(model_info, f, ensure_ascii=False, indent=2)

    return {
        'embeddings_path': OUTPUT_DIR / 'embeddings.npy',
        'vectors_path': OUTPUT_DIR / 'vectors.npy',
        'frame_mapping_path': OUTPUT_DIR / 'frame_mapping.csv',
        'frame_ids_path': OUTPUT_DIR / 'frame_ids.txt',
        'annotations_jsonl_path': OUTPUT_DIR / 'annotations.jsonl',
        'annotations_json_path': OUTPUT_DIR / 'annotations.json',
        'annations_json_path': OUTPUT_DIR / 'annations.json',
        'model_info_path': OUTPUT_DIR / 'model_info.json',
        'num_annotations': len(annotations),
        'embedding_shape': tuple(embeddings.shape),
        'model_info': model_info,
    }


def run_pipeline(config):
    start = time.time()
    clean_previous_outputs()

    if not selected_video_paths:
        raise RuntimeError(f'No video files found under {dataset_dir}. Check the Kaggle download or RAW_DATA_DIR.')

    selected_items = dataset_manifest['selected_videos']
    if len(selected_items) < int(config['max_videos']):
        print(f'Warning: requested {config["max_videos"]} videos but only selected {len(selected_items)}.')

    records, upload_manifest = extract_and_upload_keyframes(selected_items, config)
    if not records:
        raise RuntimeError('No keyframes were extracted. Check OpenCV support for this video codec.')

    embedder = ImageEmbedder(config)
    image_paths = [record['image_path'] for record in records]
    embeddings = embedder.encode(image_paths)

    outputs = export_outputs(records, embeddings, embedder, config)
    outputs['num_videos'] = len(selected_items)
    outputs['num_keyframes'] = len(records)
    outputs['elapsed_seconds'] = round(time.time() - start, 3)
    outputs['output_dir'] = OUTPUT_DIR
    outputs['upload_manifest'] = upload_manifest
    return outputs

## 9. Run pipeline

This cell downloads the Kaggle dataset if needed, extracts keyframes from 10 videos, uploads frames to MinIO when available, and exports vectors plus annotations.

In [ ]:
RUN_PIPELINE = True

pipeline_outputs = run_pipeline(CONFIG) if RUN_PIPELINE else None
pipeline_outputs

## 10. Validate outputs

In [ ]:
def validate_outputs(output_dir):
    required = [
        'dataset_manifest.yaml',
        'embeddings.npy',
        'vectors.npy',
        'frame_mapping.csv',
        'frame_ids.txt',
        'annotations.jsonl',
        'annotations.json',
        'annations.json',
        'model_info.json',
        'upload_manifest.json',
    ]
    missing_files = [name for name in required if not (output_dir / name).exists()]
    if missing_files:
        raise FileNotFoundError(f'Missing output files: {missing_files}')

    embeddings = np.load(output_dir / 'embeddings.npy')
    mapping = pd.read_csv(output_dir / 'frame_mapping.csv')
    with (output_dir / 'annotations.jsonl').open('r', encoding='utf-8') as f:
        annotation_count = sum(1 for _ in f)

    assert embeddings.dtype == np.float32
    assert len(mapping) == embeddings.shape[0] == annotation_count
    assert mapping['frame_id'].is_unique

    return {
        'embedding_shape': tuple(embeddings.shape),
        'num_mapping_rows': len(mapping),
        'num_annotations': annotation_count,
        'num_uploaded_rows': int(mapping['minio_uri'].notna().sum()) if 'minio_uri' in mapping else 0,
    }


validation_report = validate_outputs(OUTPUT_DIR) if RUN_PIPELINE else None
validation_report

## 11. Quick preview

In [ ]:
if RUN_PIPELINE:
    preview_df = pd.read_csv(OUTPUT_DIR / 'frame_mapping.csv')
    display(preview_df.head(10))

    first_image = OUTPUT_DIR / preview_df.iloc[0]['image_path']
    display(Image.open(first_image))